In [1]:
import argparse
import logging
import os
import sys
import pickle

import numpy as np
import pandas as pd

# Ensure the repository root is importable when running this notebook from the spationet folder.
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if os.path.isdir(os.path.join(repo_root, "spationet")) and repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from spationet.model import model as model_module


In [2]:
print("Loading data MPOA...")
sample_names = [f"Bregma-{i}" for i in range(1, 13)]

corpus = pd.read_csv('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/raw/mpoa/mpoa_corpus.csv', index_col=0)

pos = pd.read_csv('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/raw/mpoa/mpoa_pos.csv', index_col=0)

gene_edges = pd.read_csv('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/STRING_processed/mpoa/mpoa_gene_network.csv')

weight = pd.read_csv('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/STRING_processed/mpoa/mpoa_gene_network.csv')["abs_corr"].to_numpy()



Loading data MPOA...


In [3]:
with open(f'/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/raw/mpoa/MPOA_feat.pkl', 'rb') as f:
    feat = pickle.load(f)
n_rows_per_sample = 256

with open('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/STRING_processed/mpoa/mpoa_gene_network.pkl', "rb") as f:
    M = pickle.load(f)

with open('/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/data/spatial_processed/mpoa_diff.pkl', "rb") as f:
    diff_matrix = pickle.load(f)


In [4]:
from spationet.network.update_priors import _update_xis, _update_nis

In [5]:
from spationet.model.model import train

In [6]:
n_topics=12

In [ ]:
lda = train(
        sample_features=feat,
        difference_matrices=diff_matrix,
        difference_penalty=10,
        M=M,
        weight=weight,
        n_topics=n_topics,
        n_iters=3,
        max_lda_iter=100,
        max_admm_iter=15,
        evaluate_every=5,
        n_parallel_processes=8,
        save=False,
        output_dir="./",
    )

>>> First LDA



>>> Update Xis iteration 1


>>> Warm Update LDA after xi refinement 1
    >> LDA beta change after xi refinement: 328.358016
>>> Update Xis iteration 2


In [ ]:
beta = lda.components_.copy()
gamma = lda._unnormalized_transform(feat)

columns = [f"Topic-{i}" for i in range(n_topics)]
gamma_df = pd.DataFrame(
    gamma,
    index=(feat.index),
    columns=columns,
)

beta_df = pd.DataFrame(
    lda.components_,
    columns=feat.columns,
    index=columns,
)

lda.topic_weights = gamma_df

PATH_TO_MODELS = "/Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/example/output/mpoa/"
path_to_train_model = '_'.join((f'{PATH_TO_MODELS}/model',
                                    f'topics={n_topics}')) + '.pkl'
with open(path_to_train_model, "wb") as f:
    pickle.dump(lda, f)

gamma = lda.topic_weights

gamma_df = pd.DataFrame(gamma)
pixel_ids = [index[1] for index in feat.index]
gamma_df.index = pixel_ids 

path_to_gamma = '_'.join((f'{PATH_TO_MODELS}/gamma',
                                    f'topics={n_topics}'
                                    )) + '.csv'
gamma_df.to_csv(path_to_gamma, index=True)
print("gamma saved to:", path_to_gamma)

beta = lda.components_
beta_df = pd.DataFrame(beta)
path_to_beta = '_'.join((f'{PATH_TO_MODELS}/beta',
                                    f'topics={n_topics}'
                                    )) + '.csv'
beta_df.to_csv(path_to_beta, index=True)
print("beta saved to:", path_to_beta)

gamma saved to: /Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/example/output/mpoa//gamma_topics=12.csv
beta saved to: /Users/phuong/Library/CloudStorage/OneDrive-MichiganStateUniversity/2. SpatioNet/example/output/mpoa//beta_topics=12.csv
